# 펭귄 데이터셋 직접 EDA 해보기

**펭귄의 몸 크기와 모양은 종에 따라 어떻게 다를까요?**

Iris에서 살펴본 방법을 펭귄 데이터에 적용합니다. 결측치 처리는 강사와 함께 진행하고, 이후에는 AI를 활용해 직접 해봅니다.

**진행 순서:** 준비 → 결측치 함께 처리 → 기초 통계와 분포 → 상관관계와 산점도 → 자유 탐색

## 1. 어떤 데이터인가요?

펭귄의 신체 측정값과 종, 성별, 관찰된 섬을 담은 데이터입니다. 이번에는 측정값의 분포와 종별 차이를 탐색합니다.

**한 행은 펭귄 한 마리를 관찰한 기록입니다.**

| 컬럼 | 의미 | 단위 |
|---|---|---|
| species | 펭귄 종: Adelie, Chinstrap, Gentoo | 범주 |
| island | 관찰된 섬: Biscoe, Dream, Torgersen | 범주 |
| bill_length_mm | 부리 길이 | mm |
| bill_depth_mm | 부리 두께(위아래 두께) | mm |
| flipper_length_mm | 날개(지느러미) 길이 | mm |
| body_mass_g | 체중 | g |
| sex | 성별: Male, Female | 범주 |

species는 우리가 구분하려는 대상이고, 나머지 항목은 신체 측정값과 관찰 정보입니다.

![penguin](penguin.png)

## 2. 실습 준비

아래 준비 셀을 먼저 실행하세요. 표는 `df`라는 이름으로 불러오며, 한글 그래프 설정도 함께 준비합니다.

데이터 출처: [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) · [수업에서 사용하는 seaborn CSV](https://github.com/mwaskom/seaborn-data/blob/master/penguins.csv)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display

sns.set_style("whitegrid")
available_fonts = {font.name for font in fm.fontManager.ttflist}
korean_font = next((name for name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                    if name in available_fonts), None)
if korean_font:
    plt.rcParams["font.family"] = korean_font
else:
    print("한글 글꼴을 찾지 못했습니다.")
plt.rcParams["axes.unicode_minus"] = False

df_raw = pd.read_csv("penguin.csv")
df = df_raw.copy()
display(df.head())

## 3. 결측치란

결측치는 측정되지 않았거나 기록되지 않은 값이며, 표에서는 `NaN`으로 보입니다. 데이터 분석에 앞서 결측치를 처리해야 하는 이유는 다음과 같습니다.

#### 1. 계산 결과가 왜곡된다
* 평균, 합계, 상관관계 같은 통계가 결측치가 빠진 일부 데이터만으로 계산됩니다. 그래서 실제 모습과 달라질 수 있습니다.
* 예를 들어 센서 100개 중 30개가 비어 있으면, 평균은 나머지 70개만으로 구한 값입니다. 빠진 30개가 특정 조건(고온, 불량 등)에 몰려 있었다면 결과가 한쪽으로 치우칩니다.

#### 2. 오류가 나거나 도구마다 결과가 다릅니다
* 머신러닝 모델은 대부분 빈 값이 있으면 에러를 내며 멈춥니다.
* 분석 도구에 따라 전체를 결측으로 처리하기도 하고 결측을 제외하기도 하고 기준이 전부 다릅니다. 그래서 처리 방식을 직접 정해 두는 편이 안전합니다.

#### 3. 시각화가 잘못 보일 수 있습니다.
* 그래프에서 일부분이 사라지거나 선이 끊겨 보일 수 있습니다.
* 이를 모르고 보면 "그 시간대에는 값이 0이었다"고 오해할 수 있습니다.

#### 4. 결측 자체가 정보일 수 있습니다.
* 반도체 공정이라면 설비 점검 중이거나 센서가 고장 났을 때 값이 비어 있을 수 있습니다.
* 그래서 무조건 지우거나 채우기 전에 **왜 비었는지**를 먼저 확인하는 것이 좋습니다.

우선 데이터를 살펴보겠습니다.

In [4]:
print(f"전체: {len(df)}행, {len(df.columns)}열")
display(df.info())
display(df.isna().sum().rename("결측치 수"))

전체: 344행, 7열
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


None

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
Name: 결측치 수, dtype: int64

확인해 보면 `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex` 5개 컬럼에서 결측치가 발견되었습니다.

결측치는 처리하는 방법이 정해져 있는 것이 아니라 데이터를 살펴보고 결정하는 것입니다. 우선 결측 데이터를 확인해 보도록 하겠습니다.

In [5]:
display(df[df.isna().any(axis=1)])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


### 결측치 처리 방법

| 방법 | 언제 쓰나 |
|---|---|
| 삭제 (행/열 제거) | 결측이 적고, 무작위로 흩어져 있을 때 |
| 평균/중앙값으로 대체 | 수치형 데이터에서 간단하게 채울 때 |
| 앞/뒤 값으로 채우기 | 시간 순서가 있는 데이터일 때 |
| 그대로 두기 | 결측 자체가 의미 있거나 도구가 알아서 처리해 줄 때 |

**질문** : 이번 데이터에서는 어떤 결측 방법을 사용하는게 좋을까요? 직접 생각해 봅시다.

## 4. AI로 분석 시작하기

OpenCode를 활용해 데이터 분석을 수행해 보도록 하겠습니다! Visual Studio Code에서 OpenCode를 활용하는 방법은,

1. ctrl+shift+` 로 터미널 열기
2. 터미널에 opencode 입력
3. /models 명령어로 pro 또는 fast 모델 선택 (빠른 답변 받기 위해)
4. 대화 시작

AI에게 내가 분석 중인 코드와 데이터에 대한 정보를 알려줘야 AI가 답을 정확히 할 수 있습니다. 아래와 같이 채팅에 입력해 봅시다.

```
@"1. EDA/2. EDA_practice.ipynb" 여기서 펭귄 데이터 EDA를 진행 중이야.
노트북 코드는 코드 작성에 참고만 하고 너가 직접 입력하거나 수정하지는 마.
내가 요청한 것에 대해서만 코드를 작성해주면 내가 붙여넣기 할게. 다음 단계는 내가 요청할 때만 제시해줘.
```

이제 AI에게 결측값을 처리하는 코드를 요청해 봅시다. 앞서 어떤 행에서 결측치가 있었는지 확인해서, 각 행마다 결측치를 어떻게 처리할 것인지를 AI에게 전달합니다.

예시)
```
df의 결측치를 처리하려고 해.
bill_length_mm 컬럼의 결측치는 평균값으로 대체해줘...
```

## 5. 기초 통계와 분포 살펴보기

이제 데이터의 기초 통계를 확인해 봅시다.

```
df의 bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g에 대해 평균, 중앙값, 최솟값, 최댓값을 표로 보여줘. 부리 길이·두께와 지느러미 길이는 mm, 체중은 g이야.
```

### 직접 탐색 · 숫자를 분포로 확인하기

통계표에서 관심 있는 항목 하나를 골라 **히스토그램을 그리는 코드**를 자신의 말로 요청하세요. 가로축은 선택한 측정값, 세로축은 각 구간에 속하는 펭귄 수입니다.

값이 어디에 몰려 있고 얼마나 넓게 퍼져 있는지 살펴보세요.

종별 차이가 궁금하다면 같은 항목의 분포를 종별로 나누어 확인해도 좋습니다. 한 번 종 별로 나눠서 히스토그램을 그려서 비교해 보세요.

### 직접 탐색 · 박스플롯으로 종별 비교하기

같은 항목을 이번에는 **종별 박스플롯**으로 그리는 코드를 요청하세요. 박스플롯에서 가운데 선은 중앙값이고, 상자는 가운데 50%의 값이 모인 범위입니다.

세 종의 상자가 **서로 겹치는지, 떨어져 있는지** 확인하세요. 이 항목만으로 종을 구분할 수 있을지 생각해 봅니다.

만약 이상치가 존재한다면, 이상치는 발생할 수 있는 값일까요? 아니면 처리해야 하는 잘못된 값일까요?

## 6. 상관관계와 산점도 살펴보기

### 상관관계 히트맵

상관관계는 두 값이 함께 움직이는 경향입니다. 한쪽이 커질 때 다른 쪽도 커지면 +1에 가깝고, 다른 쪽이 작아지면 −1에 가까우며, 관계가 없으면 0에 가깝습니다. 히트맵은 이 값을 색과 숫자로 한 번에 보여줍니다.

먼저 히트맵을 그리기 전에, 네 가지 신체 측정값(부리 길이, 부리 두께, 날개 길이, 체중) 중 **관계가 있을 것 같은 쌍**을 예측해 노트북에 한 줄로 적어 봅니다.

* 예) 날개 길이와 체중은 상관관계가 있을 것 같다. 몸이 큰 펭귄일수록 날개도 길 것 같기 때문이다.

그다음 AI에게 상관관계 히트맵을 요청해 예측과 비교합니다. 예측이 틀려도 괜찮습니다. 어긋난 부분이 있다면 이유를 추측해 적어 봅니다.

* 예) 부리 길이와 부리 두께는 비례할 줄 알았는데 관계가 약했다. 종마다 부리 모양이 달라서일 수 있다.

이렇게 **가설을 세우고 데이터로 확인하는 것**이 분석과 시각화의 기본 흐름입니다.

### 두 항목을 골라 산점도로 확인하기

대각선을 제외하고 **상관계수의 절댓값이 큰 두 항목**을 골라 산점도를 요청하세요. 음수도 −1에 가까우면 강한 관계입니다. 점 하나는 펭귄 한 마리이고, 두 축은 선택한 측정값입니다.

점들이 모이는 방향과 흩어진 정도를 확인한 뒤, **종별로 점의 색을 나누도록 추가 요청**하세요. 전체에서 보인 경향이 같은 종 안에서도 나타나는지 살펴봅니다. 필요한 만큼 코드 셀을 추가해도 됩니다.

## 7. 궁금한 내용 더 확인하기

아래 세 가지를 참고하여 데이터에서 더 확인할 수 있는 정보들을 분석해 봅니다. 확인할 내용을 자신의 말로 AI에 요청하세요. 받은 코드를 실행하고 표나 그래프로 결과를 확인합니다.

### 1. 같은 종에서도 성별에 따라 체중이 다를까?

종(species)별로 암컷·수컷(sex)의 체중(body_mass_g)을 비교하세요.

- **확인할 내용:** 종과 성별로 나눈 평균·중앙값·마릿수, 체중 분포
- **분석 방법:** 그룹별 요약표와 상자그림. 상자그림은 가운데 선으로 중앙값을, 상자로 가운데 50%의 값이 모인 범위를 보여줍니다.
- 평균 차이와 함께 분포가 얼마나 겹치는지도 살펴보세요.

### 2. 펭귄 전체에서 보인 상관관계가 종별로도 비슷할까?

앞의 산점도에서 선택한 두 측정값으로 이어서 분석하세요.

- **확인할 내용:** 전체 상관계수와 각 종 안에서 계산한 상관계수의 차이
- **분석 방법:** 전체·종별 상관계수와 분석에 사용한 마릿수를 한 표로 비교
- 관계의 방향과 강도가 비슷한지 확인하고, 앞에서 그린 종별 색상 산점도와 함께 해석하세요.

### 3. 섬마다 펭귄 종의 구성이 다를까?

섬(island)별로 어떤 종(species)이 몇 마리씩 기록되어 있는지 확인하세요.

- **확인할 내용:** 섬별 종의 마릿수와 각 섬 안에서 종별로 차지하는 비율
- **분석 방법:** 섬×종 개수표와 섬별 종 비율 막대그래프
- 섬마다 전체 기록 수가 다르므로 개수와 비율을 함께 비교하세요. 여기서의 비율은 **이 데이터에 기록된 펭귄의 구성**이며, 섬 전체의 실제 개체 수 비율을 뜻하지 않습니다.

AI에게 해석을 물을 때는 실행 결과를 함께 전달하고, 설명이 실제 표·그래프와 맞는지 확인하세요. 상관관계가 원인을 뜻하거나, 종별로 점이 나뉘어 보인다고 예측 성능이 검증된 것은 아닙니다.